# M4 · Casos adversariales: ¿el lector inventa o avisa?

Corre el VLM (`Qwen/Qwen2-VL-2B-Instruct`, el mismo lector de la app) sobre imágenes del set de
los 15 casos de M3 con texto tapado (sombra casi negra, borde recortado, hoja fuera de cuadro) y
evalúa cada reto con su criterio declarado (pasa / no pasa).

- **Retos:** `M4/evaluacion_pareada/adversariales_m3.csv` (8 imágenes de `sombra_severo`,
  `rotacion_recorte_severo` y `perspectiva_severo`, elegidas por el porcentaje de texto oculto
  que registra `manifest.json`).
- **Imágenes:** las del set generado en Drive (`PROJECT_ROOT/M4/data/imagenes/`). No se suben al
  repo; se regeneran iguales con `python M4/eval_visual/generar_set.py` (semilla 42).
- **Criterio:** `M4/evaluacion_pareada/evaluar_adversariales.py` compara cada transcripción con el
  texto original del caso (`M4/eval_set/eval_set_casos.jsonl`). No pasa si aparece una frase que
  no está en la nota; pasa si el VLM activa la válvula o transcribe solo lo visible.
- **Requiere GPU (T4).** No usa Groq.

## 1 · Clonar repositorio e instalar dependencias

In [ ]:
RAMA = "Luis"

import os
if not os.path.exists("/content/repo"):
    !git clone --branch {RAMA} --single-branch https://github.com/luisNP21/Topicos-IA.git /content/repo
%cd /content/repo
!git fetch origin {RAMA}
!git checkout {RAMA}
!git pull origin {RAMA}
!pip install -r M4/vision/requirements.txt -q
!pip install "torchao>=0.16.0" -q

## 2 · Montar Drive y comprobar los retos

Verifica que existan en Drive los CSV de cada nivel y las 8 imágenes de los retos.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import csv
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
REPO = Path("/content/repo")
with open(REPO / ".env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")

IMG_DIR = PROJECT_ROOT / "M4/data/imagenes"
CASOS_ADV = REPO / "M4/evaluacion_pareada/adversariales_m3.csv"
retos = list(csv.DictReader(open(CASOS_ADV, encoding="utf-8")))
NIVELES = sorted({r["nivel"] for r in retos})

print(f"Retos: {len(retos)} | niveles: {NIVELES}")
for nivel in NIVELES:
    p = IMG_DIR / "casos" / f"{nivel}.csv"
    print(f"  CSV {nivel}: {'OK' if p.exists() else 'FALTA'} -- {p}")
faltan = [r["imagen"] for r in retos if not (IMG_DIR / r["imagen"]).exists()]
print("Imágenes de los retos:", "todas OK" if not faltan else f"FALTAN {faltan}")
assert not faltan, "Faltan imágenes en Drive: regenerar el set con M4/ejecucion/start_eval_visual.ipynb"

## 3 · Transcribir los niveles con el VLM

Transcribe los 15 casos de cada nivel con `M4/vision/run_vision.py` y deja un caché por nivel en
`PROJECT_ROOT/M4/vision/predictions/transcripciones_vlm_<nivel>.json`. Con `REUSAR = True`, si el
caché de un nivel ya existe no se vuelve a correr el VLM.

In [ ]:
# --- EDITAR AQUÍ ---
REUSAR = True
# -------------------

CACHES = {n: PROJECT_ROOT / f"M4/vision/predictions/transcripciones_vlm_{n}.json" for n in NIVELES}
%cd /content/repo
for nivel, cache in CACHES.items():
    if REUSAR and cache.exists():
        print(f"{nivel}: se reutiliza {cache.name}")
        continue
    print(f"=== {nivel} ===")
    !python M4/vision/run_vision.py --config M4/vision/config.yaml --backend vlm --project-root "{PROJECT_ROOT}" --manifest "M4/data/imagenes/casos/{nivel}.csv" 2>&1 | tail -n 30
    print("Caché:", cache, "| existe:", cache.exists())

## 4 · Evaluar los retos (pasa / no pasa)

El resultado queda en `PROJECT_ROOT/M4/evaluacion_pareada/adversariales_resultado.csv`.

In [ ]:
SALIDA = PROJECT_ROOT / "M4/evaluacion_pareada/adversariales_resultado.csv"
SALIDA.parent.mkdir(parents=True, exist_ok=True)
args_caches = " ".join(f'"{c}"' for c in CACHES.values())

%cd /content/repo
!python M4/evaluacion_pareada/evaluar_adversariales.py --casos M4/evaluacion_pareada/adversariales_m3.csv --gold M4/eval_set/eval_set_casos.jsonl --transcripciones {args_caches} --salida "{SALIDA}"

## 5 · Revisar las lecturas

Para cada reto muestra el resultado y la transcripción completa del VLM, para revisar a mano qué
leyó (sobre todo en los que no pasan).

In [ ]:
import json

resultado = list(csv.DictReader(open(SALIDA, encoding="utf-8")))
transcripciones = {}
for c in CACHES.values():
    transcripciones.update(json.loads(c.read_text(encoding="utf-8")))

for r in resultado:
    t = transcripciones.get(r["id"], {})
    print("=" * 100)
    print(f"{r['id']} | {r['nivel']} | {r['tipo_criterio']} -> {r['resultado'].upper()}  ({r['detalle']})")
    print(f"Criterio: {r['criterio']}")
    print(f"Válvula: {t.get('valvula')} | segundos: {t.get('segundos')}")
    print("Transcripción:", (t.get("texto") or "")[:1200])
print("=" * 100)
print(f"{sum(r['resultado'] == 'pasa' for r in resultado)}/{len(resultado)} retos pasan")

## 6 · Descargar los resultados

Descarga un zip con el resultado y los 3 cachés de transcripción, para versionarlos en
`M4/evaluacion_pareada/resultados/`.

In [ ]:
import shutil
from google.colab import files

dest = Path("/content/adversariales")
dest.mkdir(exist_ok=True)
shutil.copy2(SALIDA, dest / "adversariales_resultado.csv")
for nivel, c in CACHES.items():
    shutil.copy2(c, dest / f"transcripciones_vlm_{nivel}.json")
shutil.make_archive("/content/adversariales_m4", "zip", dest)
files.download("/content/adversariales_m4.zip")